# La Jolla Cove Species Identification — DINOv2 Baseline

Image classification model using transfer learning with DINOv2 (ViT-B/14) on an iNaturalist-derived dataset of species commonly found at La Jolla Cove.

**Approach**: Freeze the DINOv2 backbone and train a linear classification head. Set `FREEZE_BACKBONE = False` in config to enable full fine-tuning.

## 1. Setup and Imports

In [ ]:
import os
import json
import copy
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast

import torchvision
from torchvision import datasets, transforms
from torchvision.utils import make_grid

from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score
from PIL import Image

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("MPS (Apple Silicon) available")

### Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR = Path("data")               # root with train/ val/ test/ subdirs
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_NAME = "dinov2_vitb14"           # options: dinov2_vits14, dinov2_vitb14, dinov2_vitl14
IMAGE_SIZE = 224
FREEZE_BACKBONE = True                 # False = full fine-tuning

# ── Training ───────────────────────────────────────────────────────────────────
BATCH_SIZE = 32
LEARNING_RATE = 1e-3                   # higher LR is fine when only head is trained
LR_FINETUNE = 1e-5                     # used when FREEZE_BACKBONE is False
NUM_EPOCHS = 30
EARLY_STOP_PATIENCE = 5

# ── Hardware ───────────────────────────────────────────────────────────────────
NUM_WORKERS = 4
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"        # mixed precision only on CUDA

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"AMP enabled: {USE_AMP}")
print(f"Backbone frozen: {FREEZE_BACKBONE}")
print(f"Effective LR: {LEARNING_RATE if FREEZE_BACKBONE else LR_FINETUNE}")

## 2. Dataset Loading and Inspection

In [ ]:
# Load datasets with a basic ToTensor transform just for inspection.
# Real augmentation transforms are defined in Section 3.
_inspect_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset_raw = datasets.ImageFolder(DATA_DIR / "train", transform=_inspect_transform)
val_dataset_raw   = datasets.ImageFolder(DATA_DIR / "val",   transform=_inspect_transform)
test_dataset_raw  = datasets.ImageFolder(DATA_DIR / "test",  transform=_inspect_transform)

class_names = train_dataset_raw.classes
num_classes = len(class_names)

print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}\n")

for split_name, ds in [("Train", train_dataset_raw), ("Val", val_dataset_raw), ("Test", test_dataset_raw)]:
    targets = [s[1] for s in ds.samples]
    counts = Counter(targets)
    print(f"  {split_name}: {len(ds)} images")
    for idx in sorted(counts):
        print(f"    {ds.classes[idx]:>30s}: {counts[idx]}")
    print()

### Sample images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
indices = random.sample(range(len(train_dataset_raw)), 10)
for ax, idx in zip(axes.flat, indices):
    img, label = train_dataset_raw[idx]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(class_names[label], fontsize=9)
    ax.axis("off")
fig.suptitle("Sample Training Images", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Transforms and DataLoaders

In [ ]:
# ImageNet normalization (used by DINOv2)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_test_transforms = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),   # 256 for IMAGE_SIZE=224
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
train_dataset = datasets.ImageFolder(DATA_DIR / "train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATA_DIR / "val",   transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(DATA_DIR / "test",  transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

## 4. Model

Load a pretrained DINOv2 backbone from `torch.hub` and attach a linear classification head. The backbone is frozen by default (linear probing); set `FREEZE_BACKBONE = False` for full fine-tuning.

In [ ]:
class DINOv2Classifier(nn.Module):
    """DINOv2 backbone + linear classification head."""

    def __init__(self, backbone, embed_dim: int, num_classes: int, freeze_backbone: bool = True):
        super().__init__()
        self.backbone = backbone
        self.backbone_frozen = freeze_backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            self.backbone.eval()
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):
        if self.backbone_frozen:
            with torch.no_grad():
                features = self.backbone(x)
        else:
            features = self.backbone(x)
        return self.head(features)


class StaticPTQWrapper(nn.Module):
    """Float backbone + QuantStub → head (static PTQ) → DeQuantStub.

    INT8 weights and calibrated activation ranges apply to the classification head;
    the ViT backbone stays float32 (full-graph static PTQ is not used here).
    """

    def __init__(self, inner: DINOv2Classifier):
        super().__init__()
        self.backbone = inner.backbone
        self.backbone_frozen = inner.backbone_frozen
        self.quant = torch.ao.quantization.QuantStub()
        self.head = inner.head
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        if self.backbone_frozen:
            with torch.no_grad():
                features = self.backbone(x)
        else:
            features = self.backbone(x)
        features = self.quant(features)
        out = self.head(features)
        return self.dequant(out)

In [ ]:
# DINOv2 embed dims: vits14=384, vitb14=768, vitl14=1024
EMBED_DIMS = {"dinov2_vits14": 384, "dinov2_vitb14": 768, "dinov2_vitl14": 1024}

backbone = torch.hub.load("facebookresearch/dinov2", MODEL_NAME)
embed_dim = EMBED_DIMS[MODEL_NAME]

model = DINOv2Classifier(backbone, embed_dim, num_classes, freeze_backbone=FREEZE_BACKBONE)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Frozen parameters:    {total_params - trainable_params:>12,}")

## 5. Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss()

lr = LEARNING_RATE if FREEZE_BACKBONE else LR_FINETUNE
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=lr, weight_decay=1e-2)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-7)

scaler = GradScaler("cuda", enabled=USE_AMP)

print(f"Loss:      CrossEntropyLoss")
print(f"Optimizer: AdamW (lr={lr}, wd=1e-2)")
print(f"Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})")
print(f"Scaler:    {'enabled' if USE_AMP else 'disabled'}")

## 6. Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, use_amp):
    model.train()
    # Keep backbone in eval mode when frozen (batchnorm / dropout)
    if FREEZE_BACKBONE:
        model.backbone.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device, use_amp):
    model.eval()
    running_loss = 0.0
    correct_top1 = 0
    correct_top3 = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        with autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        total += labels.size(0)

        _, preds_top1 = outputs.max(1)
        correct_top1 += preds_top1.eq(labels).sum().item()

        # Top-3 accuracy
        _, preds_top3 = outputs.topk(min(3, outputs.size(1)), dim=1)
        correct_top3 += preds_top3.eq(labels.unsqueeze(1)).any(1).sum().item()

    return running_loss / total, correct_top1 / total, correct_top3 / total

In [ ]:
history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [], "val_top3_acc": [],
}

best_val_acc = 0.0
best_model_wts = None
epochs_no_improve = 0

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Val Loss':>10}  {'Train Acc':>10}  {'Val Acc':>10}  {'Val Top3':>10}  {'LR':>10}  {'Time':>6}")
print("-" * 85)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE, USE_AMP)
    val_loss, val_acc, val_top3 = evaluate(model, val_loader, criterion, DEVICE, USE_AMP)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_top3_acc"].append(val_top3)

    elapsed = time.time() - t0
    current_lr = scheduler.get_last_lr()[0]

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        marker = " *"
    else:
        epochs_no_improve += 1

    print(f"{epoch:>5d}  {train_loss:>10.4f}  {val_loss:>10.4f}  {train_acc:>10.4f}  {val_acc:>10.4f}  {val_top3:>10.4f}  {current_lr:>10.2e}  {elapsed:>5.1f}s{marker}")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs)")
        break

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

### Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_range, history["train_loss"], label="Train")
ax1.plot(epochs_range, history["val_loss"], label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], label="Train")
ax2.plot(epochs_range, history["val_acc"], label="Val Top-1")
ax2.plot(epochs_range, history["val_top3_acc"], label="Val Top-3", linestyle="--")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

fig.suptitle("Training Curves", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved training curves to {OUTPUT_DIR / 'training_curves.png'}")

## 7. Evaluation on Test Set

In [ ]:
# Restore best model weights
model.load_state_dict(best_model_wts)
model.eval()

test_loss, test_top1, test_top3 = evaluate(model, test_loader, criterion, DEVICE, USE_AMP)
print(f"Test Loss:         {test_loss:.4f}")
print(f"Test Top-1 Acc:    {test_top1:.4f}")
print(f"Test Top-3 Acc:    {test_top3:.4f}")

### Classification report and confusion matrix

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device, use_amp):
    """Collect all predictions and true labels from a DataLoader."""
    all_labels = []
    all_preds = []
    all_probs = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        with autocast("cuda", enabled=use_amp):
            outputs = model(images)
        probs = torch.softmax(outputs.float(), dim=1)
        all_labels.append(labels)
        all_preds.append(outputs.argmax(1).cpu())
        all_probs.append(probs.cpu())

    return (torch.cat(all_labels).numpy(),
            torch.cat(all_preds).numpy(),
            torch.cat(all_probs).numpy())

y_true, y_pred, y_probs = collect_predictions(model, test_loader, DEVICE, USE_AMP)

print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

# Adaptive figure size based on number of classes
fig_size = max(10, num_classes * 0.4)
fig, ax = plt.subplots(figsize=(fig_size, fig_size))

sns.heatmap(cm, annot=num_classes <= 30, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax,
            linewidths=0.5, linecolor="white")
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("Confusion Matrix — Test Set", fontsize=14)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

# Also show sklearn top-k accuracy using probability matrix
if num_classes >= 3:
    sk_top3 = top_k_accuracy_score(y_true, y_probs, k=3, labels=range(num_classes))
    print(f"sklearn top-3 accuracy (verification): {sk_top3:.4f}")

## 8. Prediction on a Single Image

In [ ]:
@torch.no_grad()
def predict_single_image(image_path, model, transform, class_names, device, top_k=3):
    """Run inference on a single image and return top-k predictions."""
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with autocast("cuda", enabled=USE_AMP):
        logits = model(img_tensor)
    probs = torch.softmax(logits.float(), dim=1).squeeze(0)

    top_probs, top_indices = probs.topk(top_k)
    results = [(class_names[idx], prob.item()) for idx, prob in zip(top_indices, top_probs)]
    return img, results


# Pick a random test image for demonstration
sample_path, sample_label = test_dataset.samples[random.randint(0, len(test_dataset) - 1)]
true_name = class_names[sample_label]

img, predictions = predict_single_image(sample_path, model, val_test_transforms, class_names, DEVICE, top_k=3)

fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.5]})

ax_img.imshow(img)
ax_img.set_title(f"True: {true_name}", fontsize=12)
ax_img.axis("off")

names = [p[0] for p in predictions]
probs = [p[1] for p in predictions]
colors = ["#2ecc71" if n == true_name else "#3498db" for n in names]
ax_bar.barh(range(len(names)), probs, color=colors)
ax_bar.set_yticks(range(len(names)))
ax_bar.set_yticklabels(names, fontsize=10)
ax_bar.set_xlabel("Probability")
ax_bar.set_title("Top-3 Predictions")
ax_bar.set_xlim(0, 1)
ax_bar.invert_yaxis()

plt.tight_layout()
plt.show()

for rank, (name, prob) in enumerate(predictions, 1):
    print(f"  #{rank}: {name:<30s} {prob:.4f}")

## 9. Saving Artifacts

In [ ]:
# ── 1. Best model checkpoint ───────────────────────────────────────────────────
checkpoint_path = OUTPUT_DIR / "best_model.pth"
torch.save({
    "model_state_dict": best_model_wts,
    "model_name": MODEL_NAME,
    "num_classes": num_classes,
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "freeze_backbone": FREEZE_BACKBONE,
    "best_val_acc": best_val_acc,
}, checkpoint_path)
print(f"Saved model checkpoint: {checkpoint_path}")

# ── 2. Class names ─────────────────────────────────────────────────────────────
class_names_path = OUTPUT_DIR / "class_names.json"
with open(class_names_path, "w") as f:
    json.dump(class_names, f, indent=2)
print(f"Saved class names:     {class_names_path}")

# ── 3. Metrics ─────────────────────────────────────────────────────────────────
metrics = {
    "best_val_acc": best_val_acc,
    "test_top1_acc": test_top1,
    "test_top3_acc": test_top3,
    "test_loss": test_loss,
    "num_epochs_trained": len(history["train_loss"]),
    "model_name": MODEL_NAME,
    "freeze_backbone": FREEZE_BACKBONE,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "learning_rate": lr,
    "history": history,
}
metrics_path = OUTPUT_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics:         {metrics_path}")

# ── 4. Training curves image was already saved in Section 6 ────────────────────
print(f"Training curves:       {OUTPUT_DIR / 'training_curves.png'}")
print("\nAll artifacts saved.")

## 10. Post-training quantization & TFLite (Arduino UNO Q)

**PyTorch**: **static** post-training quantization on the **classification head** only (`QuantStub` → `LayerNorm` + `Linear` → `DeQuantStub`), with calibration batches from `val_loader`. That gives **INT8 weights and calibrated activation ranges** on the head matmuls; the DINOv2 backbone stays **float32** (eager-mode static quant of the full ViT is not supported here).

**TFLite**: ONNX → **onnx2tf** (default `flatbuffer_direct`) usually writes `model_float32.tflite` / `model_float16.tflite` **without** a `saved_model.pb` on disk. The helper only calls TensorFlow’s `TFLiteConverter.from_saved_model` when a SavedModel is actually present; otherwise it uses onnx2tf’s `.tflite` files. Conversion runs in a **subprocess** so TensorFlow/OpenCV are not imported in the same kernel as PyTorch (avoids NumPy docstring conflicts).

Install converters once: `pip install onnx onnx2tf tensorflow`

TFLite conversion is implemented in **`export_tflite_subprocess.py`** (called from the next cell) so the notebook never imports TensorFlow inline.

Run after training (or ensure `outputs/best_model.pth` exists from a prior run).

In [66]:
# ── Float model on CPU (from in-memory training run or best_model.pth) ─────────
def _load_float_classifier_cpu():
    if "best_model_wts" in globals() and best_model_wts is not None and "model" in globals():
        m = copy.deepcopy(model).cpu().eval()
        return m
    ckpt = torch.load(OUTPUT_DIR / "best_model.pth", map_location="cpu", weights_only=False)
    name = ckpt["model_name"]
    n_cls = ckpt["num_classes"]
    fb = ckpt.get("freeze_backbone", True)
    backbone = torch.hub.load("facebookresearch/dinov2", name)
    m = DINOv2Classifier(backbone, EMBED_DIMS[name], n_cls, freeze_backbone=fb)
    m.load_state_dict(ckpt["model_state_dict"])
    return m.eval()


model_float_cpu = _load_float_classifier_cpu()

# ── Post-training static quantization: head uses INT8 weights + calibrated activations ─
# (ViT backbone stays float32; dynamic quant alone only INT8-compresses weights, not activations.)
def _pick_quantization_backend() -> str:
    try:
        torch.backends.quantized.engine = "fbgemm"
        return "fbgemm"
    except RuntimeError:
        torch.backends.quantized.engine = "qnnpack"
        return "qnnpack"


_qbackend = _pick_quantization_backend()
_qcfg = torch.ao.quantization.get_default_qconfig(_qbackend)

_wrapped = StaticPTQWrapper(model_float_cpu).eval()
_wrapped.qconfig = None
_wrapped.backbone.qconfig = None
_wrapped.quant.qconfig = _qcfg
_wrapped.dequant.qconfig = _qcfg
_wrapped.head.qconfig = _qcfg
for _child in _wrapped.head:
    _child.qconfig = _qcfg

model_ptq = torch.ao.quantization.prepare(_wrapped, inplace=False)
model_ptq.eval()
with torch.no_grad():
    if "val_loader" in globals():
        for images, _ in val_loader:
            model_ptq(images.cpu())
    else:
        model_ptq(torch.randn(8, 3, IMAGE_SIZE, IMAGE_SIZE))

model_ptq = torch.ao.quantization.convert(model_ptq, inplace=False)
model_ptq.eval()

ptq_path = OUTPUT_DIR / "best_model_ptq.pth"
_ckpt_fp = OUTPUT_DIR / "best_model.pth"
if _ckpt_fp.exists():
    _src = torch.load(_ckpt_fp, map_location="cpu", weights_only=False)
    meta = {k: v for k, v in _src.items() if k != "model_state_dict"}
else:
    meta = {
        "model_name": MODEL_NAME,
        "num_classes": num_classes,
        "class_names": class_names,
        "image_size": IMAGE_SIZE,
        "freeze_backbone": FREEZE_BACKBONE,
        "best_val_acc": globals().get("best_val_acc"),
    }

torch.save({
    **meta,
    "model_state_dict": model_ptq.state_dict(),
    "quantization": "static_int8_head",
    "qengine": _qbackend,
}, ptq_path)
print(f"Saved PTQ checkpoint: {ptq_path}")

# ── ONNX export (float model — reliable path for ViT → TFLite) ────────────────
onnx_path = OUTPUT_DIR / "model.onnx"
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
torch.onnx.export(
    model_float_cpu,
    dummy,
    onnx_path,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
    do_constant_folding=True,
)
print(f"Exported ONNX: {onnx_path}")

# ── ONNX → TFLite: run in separate Python process (see export_tflite_subprocess.py) ─
import subprocess
import sys
from pathlib import Path


def _find_export_script() -> Path:
    for root in (Path.cwd(), Path.cwd() / "species_identification"):
        p = root / "export_tflite_subprocess.py"
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(
        "export_tflite_subprocess.py not found. "
        "It should live in species_identification/ next to this notebook."
    )


saved_model_dir = OUTPUT_DIR / "saved_model_tf"
saved_model_dir.mkdir(parents=True, exist_ok=True)

calib_arg = "none"
calib_path = OUTPUT_DIR / "_tflite_calib.npz"
if "val_loader" in globals():
    try:
        arrs = []
        n = 0
        for images, _ in val_loader:
            x = images.cpu().numpy()
            for i in range(x.shape[0]):
                arrs.append(x[i : i + 1])
                n += 1
                if n >= 100:
                    break
            if n >= 100:
                break
        if arrs:
            np.savez(calib_path, *arrs)
            calib_arg = str(calib_path.resolve())
    except Exception as ex:
        print("Optional INT8 calibration not saved:", ex)

_export = _find_export_script()
subprocess.check_call(
    [
        sys.executable,
        str(_export),
        str(onnx_path.resolve()),
        str(saved_model_dir.resolve()),
        calib_arg,
    ]
)

print(
    "Done. Copy a .tflite file to the Arduino UNO Q and load it with the TensorFlow Lite "
    "Python runtime or tflite_runtime."
)


Saved PTQ checkpoint: outputs\best_model_ptq.pth
Exported ONNX: outputs\model.onnx
Done. Copy a .tflite file to the Arduino UNO Q and load it with the TensorFlow Lite Python runtime or tflite_runtime.


In [65]:
model

DINOv2Classifier(
  (backbone): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
        (ls2): LayerScale()
        (drop_path2)

### Evaluate PTQ model on the test set

**Static** INT8 on the **classification head** (weights + activations via calibration); backbone stays float. Runs on **CPU**. Uses in-memory `model_ptq` from the export cell when present; otherwise reloads `outputs/best_model.pth` + `outputs/best_model_ptq.pth` into `StaticPTQWrapper`. Requires `test_loader`, `evaluate`, `DINOv2Classifier`, `StaticPTQWrapper`, and `EMBED_DIMS` from earlier sections.

In [67]:
def _pick_quantization_backend_eval() -> str:
    try:
        torch.backends.quantized.engine = "fbgemm"
        return "fbgemm"
    except RuntimeError:
        torch.backends.quantized.engine = "qnnpack"
        return "qnnpack"


def _load_static_ptq_from_disk():
    fp_path = OUTPUT_DIR / "best_model.pth"
    ptq_path = OUTPUT_DIR / "best_model_ptq.pth"
    if not fp_path.is_file() or not ptq_path.is_file():
        raise FileNotFoundError(
            "Need outputs/best_model.pth and outputs/best_model_ptq.pth — run the quantization cell."
        )
    float_ckpt = torch.load(fp_path, map_location="cpu", weights_only=False)
    ptq_ckpt = torch.load(ptq_path, map_location="cpu", weights_only=False)
    name = float_ckpt["model_name"]
    n_cls = float_ckpt["num_classes"]
    fb = float_ckpt.get("freeze_backbone", True)
    backbone = torch.hub.load("facebookresearch/dinov2", name)
    m = DINOv2Classifier(backbone, EMBED_DIMS[name], n_cls, freeze_backbone=fb)
    m.load_state_dict(float_ckpt["model_state_dict"])
    m.eval()
    qback = ptq_ckpt.get("qengine") or _pick_quantization_backend_eval()
    torch.backends.quantized.engine = qback
    qcfg = torch.ao.quantization.get_default_qconfig(qback)
    w = StaticPTQWrapper(m).eval()
    w.qconfig = None
    w.backbone.qconfig = None
    w.quant.qconfig = qcfg
    w.dequant.qconfig = qcfg
    w.head.qconfig = qcfg
    for _child in w.head:
        _child.qconfig = qcfg
    prepared = torch.ao.quantization.prepare(w, inplace=False)
    converted = torch.ao.quantization.convert(prepared, inplace=False)
    converted.load_state_dict(ptq_ckpt["model_state_dict"])
    return converted


def _model_ptq_for_eval():
    if "model_ptq" in globals() and globals().get("model_ptq") is not None:
        return globals()["model_ptq"]
    return _load_static_ptq_from_disk()


if "test_loader" not in globals() or "evaluate" not in globals():
    raise RuntimeError("Run dataset + training sections first so test_loader and evaluate() exist.")

_ptq = _model_ptq_for_eval().cpu().eval()
_cpu = torch.device("cpu")

ptq_loss, ptq_top1, ptq_top3 = evaluate(_ptq, test_loader, criterion, _cpu, use_amp=False)

print("PTQ (static INT8 head — weights + calibrated activations) — test set (CPU)")
print(f"  Test Loss:      {ptq_loss:.4f}")
print(f"  Test Top-1 Acc: {ptq_top1:.4f}")
print(f"  Test Top-3 Acc: {ptq_top3:.4f}")
if "test_top1" in globals():
    print("\nFloat model (same test set, for reference)")
    print(f"  Test Top-1 Acc: {test_top1:.4f}")
    print(f"  Test Top-3 Acc: {test_top3:.4f}")
    print(f"  Δ Top-1:        {ptq_top1 - test_top1:+.4f}")

PTQ (dynamic INT8 Linear) — test set (CPU)
  Test Loss:      6.4194
  Test Top-1 Acc: 0.0142
  Test Top-3 Acc: 0.0384

Float model (same test set, for reference)
  Test Top-1 Acc: 0.9227
  Test Top-3 Acc: 0.9847
  Δ Top-1:        -0.9085
